# 🏥 Health Eat — 경구약제 이미지 EDA & 전처리 전략

> **프로젝트**: 알약 이미지 객체 검출 (Object Detection)  
> **평가 지표**: mAP@[0.75:0.95]

---

## 목차
1. [환경 설정 및 데이터 로드](#1-환경-설정-및-데이터-로드)
2. [기본 EDA — 이미지 & 어노테이션 분석](#2-기본-eda)
3. [데이터 현황 진단 (특이사항)](#3-데이터-현황-진단)
4. [전처리 전략 구현](#4-전처리-전략-구현)
5. [증강 기법 시각화](#5-증강-기법-시각화)

---
## 1. 환경 설정 및 데이터 로드

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image, ImageFilter, ImageEnhance
from collections import Counter
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. 압축 풀기 (파일 이름: ai10-level1-project.zip)
!unzip -q /ai10-level1-project.zip -d ./sprint_ai_project1_data

# 2. 경로 설정 (순서 중요: DATA_DIR을 먼저 정의해야 합니다)
# 압축 해제 후 경로가 sprint_ai_project1_data/sprint_ai_project1_data 로 중첩되어 생성되므로 수정합니다.
DATA_DIR       = Path('./sprint_ai_project1_data/sprint_ai_project1_data')
TRAIN_IMG_DIR  = DATA_DIR / 'train_images'
TRAIN_ANN_DIR  = DATA_DIR / 'train_annotations'
TEST_IMG_DIR   = DATA_DIR / 'test_images'

# 3. 시각화 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi']  = 120

# 4. 실제 경로가 잘 설정되었는지 확인하는 코드 (추가됨)
print(f"데이터 폴더 확인: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    print(f"찾은 하위 폴더: {os.listdir(DATA_DIR)}")

print('✅ 환경 설정 완료')

replace ./sprint_ai_project1_data/sprint_ai_project1_data/test_images/1.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 데이터 폴더 확인: True
찾은 하위 폴더: ['test_images', 'train_annotations', 'train_images']
✅ 환경 설정 완료


In [ ]:
def load_all_annotations(ann_dir: Path) -> dict:
    """
    여러 JSON 파일로 분산된 COCO 포맷 어노테이션을
    하나의 딕셔너리로 병합합니다.
    """
    all_images      = []
    all_annotations = []
    all_categories  = {}

    for json_file in sorted(ann_dir.glob('*.json')):
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        all_images.extend(data.get('images', []))
        all_annotations.extend(data.get('annotations', []))

        for cat in data.get('categories', []):
            all_categories[cat['id']] = cat

    return {
        'images':      all_images,
        'annotations': all_annotations,
        'categories':  list(all_categories.values())
    }

coco = load_all_annotations(TRAIN_ANN_DIR)

print(f'이미지 수      : {len(coco["images"]):,}')
print(f'어노테이션 수  : {len(coco["annotations"]):,}')
print(f'카테고리(클래스) 수: {len(coco["categories"]):,}')

이미지 수      : 0
어노테이션 수  : 0
카테고리(클래스) 수: 0


---
## 2. 기본 EDA

In [ ]:
# ── 2-1. 클래스 분포 확인 ──────────────────────────────────────────
cat_id_to_name = {c['id']: c['name'] for c in coco['categories']}
ann_df = pd.DataFrame(coco['annotations'])

class_counts = ann_df['category_id'].value_counts()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(class_counts)), class_counts.values, color='steelblue', width=0.8)
ax.set_title('클래스별 어노테이션 수 (내림차순)', fontsize=13)
ax.set_xlabel('클래스 순위')
ax.set_ylabel('어노테이션 수')
ax.axhline(class_counts.mean(), color='tomato', linestyle='--', label=f'평균: {class_counts.mean():.1f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'상위 5개 클래스:\n{class_counts.head()}')
print(f'\n하위 5개 클래스 (데이터 부족 주의):\n{class_counts.tail()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2-2. 이미지별 알약 개수 분포 ──────────────────────────────────
pills_per_image = ann_df.groupby('image_id').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 히스토그램
axes[0].hist(pills_per_image.values, bins=range(1, 7), edgecolor='white', color='mediumpurple')
axes[0].set_title('이미지당 알약 수 분포')
axes[0].set_xlabel('알약 수')
axes[0].set_ylabel('이미지 수')
axes[0].set_xticks(range(1, 6))

# 바운딩 박스 크기 분포
ann_df['bbox_area'] = ann_df['bbox'].apply(lambda b: b[2] * b[3])
axes[1].hist(ann_df['bbox_area'].values, bins=50, color='teal', edgecolor='white')
axes[1].set_title('바운딩 박스 면적 분포')
axes[1].set_xlabel('면적 (px²)')
axes[1].set_ylabel('어노테이션 수')

plt.tight_layout()
plt.show()

In [ ]:
# ── 2-3. 촬영 환경 메타데이터 분포 ────────────────────────────────
img_df = pd.DataFrame(coco['images'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

meta_cols = [
    ('back_color',  '배경색'),
    ('light_color', '조명색'),
    ('drug_dir',    '촬영 각도'),
]

for ax, (col, title) in zip(axes, meta_cols):
    if col in img_df.columns:
        counts = img_df[col].value_counts()
        ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90)
        ax.set_title(title)
    else:
        ax.text(0.5, 0.5, f'컬럼 없음:\n{col}', ha='center', va='center')
        ax.set_title(title)

plt.suptitle('⚠️ 촬영 환경 편향성 확인 — 고정 조건 비율이 높을수록 증강 필요', y=1.02, fontsize=11, color='tomato')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2-4. 샘플 이미지 + 바운딩 박스 시각화 ────────────────────────
def visualize_sample(image_id: int, n_samples: int = 4):
    img_info_list = [img for img in coco['images'] if img['id'] == image_id]
    if not img_info_list:
        print(f'image_id {image_id} 없음')
        return

    img_info = img_info_list[0]
    img_path = TRAIN_IMG_DIR / img_info['file_name']
    img      = Image.open(img_path).convert('RGB')

    anns = [a for a in coco['annotations'] if a['image_id'] == image_id]

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.imshow(img)

    colors = ['tomato', 'dodgerblue', 'limegreen', 'orange']
    for i, ann in enumerate(anns):
        x, y, w, h = ann['bbox']
        rect = patches.Rectangle((x, y), w, h,
                                  linewidth=2, edgecolor=colors[i % len(colors)], facecolor='none')
        ax.add_patch(rect)
        label = cat_id_to_name.get(ann['category_id'], str(ann['category_id']))
        ax.text(x, y - 4, label, fontsize=8, color=colors[i % len(colors)],
                bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))

    ax.set_title(f"image_id={image_id} | 알약 {len(anns)}개 | {img_info.get('dl_name', '')}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# 랜덤 샘플 4장 시각화
sample_ids = random.sample([img['id'] for img in coco['images']], 4)
for sid in sample_ids:
    visualize_sample(sid)

---
## 3. 데이터 현황 진단

EDA를 통해 확인된 3가지 핵심 문제점:

| # | 문제 | 원인 | 영향 |
|---|------|------|------|
| 1 | **환경 편향성** | 연회색 배경, 주백색 조명, 고정 각도 | 실제 사용 환경에서 인식 실패 |
| 2 | **반쪽 알약 부재** | 분할 약 이미지 전무 | 처방 조제 약 인식 불가 |
| 3 | **해상도 불균일** | 이미지마다 품질 차이 | 각인 문자 인식 불안정 |

In [ ]:
# ── 3-1. 이미지 해상도 분포 확인 ──────────────────────────────────
resolutions = []
for img_info in random.sample(coco['images'], min(200, len(coco['images']))):
    img_path = TRAIN_IMG_DIR / img_info['file_name']
    if img_path.exists():
        with Image.open(img_path) as img:
            resolutions.append(img.size)  # (width, height)

widths  = [r[0] for r in resolutions]
heights = [r[1] for r in resolutions]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('이미지 너비 분포')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=30, color='mediumpurple', edgecolor='white')
axes[1].set_title('이미지 높이 분포')
axes[1].set_xlabel('Height (px)')

plt.suptitle(f'해상도 분포 | 너비 평균: {np.mean(widths):.0f}px / 높이 평균: {np.mean(heights):.0f}px')
plt.tight_layout()
plt.show()

---
## 4. 전처리 전략 구현

### 전략 요약

```
[환경 극복]        배경 합성 / Color Jittering / 가우시안 노이즈·블러
[반쪽 알약 대응]   Random Erasing / Cutout
[정교한 인식]      Sharpening / 고해상도 리사이즈
```

In [ ]:
# ── 4-1. Albumentations 증강 파이프라인 ───────────────────────────
#
# 모델링 팀은 이 transform을 DataLoader에 그대로 연결하면 됩니다.
# 바운딩 박스 포맷: 'coco' = [x_min, y_min, width, height]

train_transform = A.Compose([

    # ① 고해상도 리사이즈 — mAP@0.75+ 기준 맞추기 위해 디테일 보존
    A.LongestMaxSize(max_size=1024),
    A.PadIfNeeded(min_height=1024, min_width=1024,
                  border_mode=cv2.BORDER_CONSTANT, value=0),

    # ② 환경 편향 극복 — 조명 / 색상 변조
    A.ColorJitter(
        brightness=0.3,   # 밝기 ±30%
        contrast=0.3,     # 대비 ±30%
        saturation=0.2,   # 채도 ±20%
        hue=0.05,         # 색조 미세 변화
        p=0.7
    ),

    # ③ 환경 편향 극복 — 노이즈 & 블러 (폰카 흔들림 시뮬레이션)
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50)),
        A.MotionBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=3),
    ], p=0.4),

    # ④ 반쪽 알약 대응 — 일부 영역 강제 삭제
    A.CoarseDropout(
        max_holes=4,
        max_height=80,
        max_width=80,
        min_holes=1,
        fill_value=0,
        p=0.4
    ),

    # ⑤ 기하학적 변환 — 회전 / 플립 (알약은 방향 무관)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.5),

    # ⑥ 정교한 인식 — 샤프닝 (각인 문자 선명도 향상)
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.3),

    # ⑦ 정규화
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),

], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['category_ids'],
    min_visibility=0.3       # bbox 30% 미만으로 잘리면 제거
))

# 검증/테스트용 (증강 없이 리사이즈 + 정규화만)
val_transform = A.Compose([
    A.LongestMaxSize(max_size=1024),
    A.PadIfNeeded(min_height=1024, min_width=1024,
                  border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']))

print('✅ 증강 파이프라인 정의 완료')
print(f'   train_transform 단계 수: {len(train_transform.transforms)}')

In [ ]:
# ── 4-2. 배경 합성 유틸 ───────────────────────────────────────────
#
# 고정 회색 배경 대신 나무 책상, 유리 테이블 등 다양한 배경을 합성합니다.
# bg_dir 에 배경 이미지들을 넣어두면 랜덤 적용됩니다.

def swap_background(pill_img: np.ndarray,
                    bg_dir: Path,
                    bg_thresh: int = 200) -> np.ndarray:
    """
    pill_img : 원본 알약 이미지 (H, W, 3) — BGR
    bg_dir   : 대체할 배경 이미지 폴더
    bg_thresh: 배경으로 판단할 밝기 임계값 (연회색 배경 기준)
    """
    bg_paths = list(bg_dir.glob('*.jpg')) + list(bg_dir.glob('*.png'))
    if not bg_paths:
        return pill_img  # 배경 없으면 원본 반환

    h, w = pill_img.shape[:2]

    # 배경 마스크 생성 (밝고 채도 낮은 영역 = 회색 배경)
    gray    = cv2.cvtColor(pill_img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, bg_thresh, 255, cv2.THRESH_BINARY)
    mask    = cv2.erode(mask, np.ones((3, 3)), iterations=1)

    # 랜덤 배경 불러와서 리사이즈
    bg = cv2.imread(str(random.choice(bg_paths)))
    bg = cv2.resize(bg, (w, h))

    # 합성
    mask_3ch = cv2.merge([mask, mask, mask])
    result   = np.where(mask_3ch == 255, bg, pill_img)
    return result

print('✅ swap_background 함수 정의 완료')
print('   사용법: result = swap_background(cv2_img, Path("./backgrounds"))')

---
## 5. 증강 기법 시각화

각 증강 기법이 실제로 이미지를 어떻게 바꾸는지 모델링 팀이 한눈에 확인할 수 있도록 시각화합니다.

In [ ]:
# ── 5-1. 증강 기법별 Before / After 비교 ─────────────────────────

def show_augmentation_grid(image_id: int):
    """하나의 원본 이미지에 각 증강 기법을 적용해 나란히 보여줍니다."""
    img_info = next(img for img in coco['images'] if img['id'] == image_id)
    img_path = TRAIN_IMG_DIR / img_info['file_name']
    img_np   = np.array(Image.open(img_path).convert('RGB'))

    augmentations = [
        ('원본',                    None),
        ('Color Jitter\n(조명 변화)', A.ColorJitter(brightness=0.4, contrast=0.4,
                                                   saturation=0.4, hue=0.1, p=1.0)),
        ('Gaussian Noise\n(노이즈)', A.GaussNoise(var_limit=(40, 80), p=1.0)),
        ('Motion Blur\n(손떨림)',    A.MotionBlur(blur_limit=9, p=1.0)),
        ('Coarse Dropout\n(반쪽 대응)', A.CoarseDropout(max_holes=6, max_height=100,
                                                       max_width=100, p=1.0)),
        ('Sharpen\n(각인 선명화)',   A.Sharpen(alpha=(0.5, 0.8), lightness=(0.8, 1.0), p=1.0)),
        ('Rotate 30°\n(각도 변화)', A.Rotate(limit=30, p=1.0)),
        ('Horizontal Flip',         A.HorizontalFlip(p=1.0)),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for ax, (title, aug) in zip(axes, augmentations):
        if aug is None:
            out = img_np
        else:
            out = A.Compose([aug])(image=img_np)['image']
        ax.imshow(out)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    plt.suptitle(f'증강 기법 비교 | image_id={image_id}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# 실행: 원하는 image_id 로 변경
show_augmentation_grid(image_id=sample_ids[0])

In [ ]:
# ── 5-2. 증강 후 바운딩 박스 유지 확인 ────────────────────────────
#
# 증강 후 bbox가 올바르게 따라오는지 검증하는 셀입니다.

def verify_bbox_after_aug(image_id: int):
    img_info = next(img for img in coco['images'] if img['id'] == image_id)
    img_path = TRAIN_IMG_DIR / img_info['file_name']
    img_np   = np.array(Image.open(img_path).convert('RGB'))

    anns     = [a for a in coco['annotations'] if a['image_id'] == image_id]
    bboxes   = [a['bbox'] for a in anns]
    cat_ids  = [a['category_id'] for a in anns]

    aug_for_bbox = A.Compose([
        A.Rotate(limit=30, p=1.0),
        A.HorizontalFlip(p=1.0),
        A.ColorJitter(brightness=0.3, contrast=0.3, p=1.0),
    ], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids'],
                                min_visibility=0.3))

    result      = aug_for_bbox(image=img_np, bboxes=bboxes, category_ids=cat_ids)
    aug_img     = result['image']
    aug_bboxes  = result['bboxes']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors    = ['tomato', 'dodgerblue', 'limegreen', 'orange']

    for ax, (image, boxes, title) in zip(axes, [
        (img_np,  bboxes,     '원본 + bbox'),
        (aug_img, aug_bboxes, '증강 후 + bbox'),
    ]):
        ax.imshow(image)
        for i, (x, y, w, h) in enumerate(boxes):
            rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                     edgecolor=colors[i % len(colors)], facecolor='none')
            ax.add_patch(rect)
        ax.set_title(title)
        ax.axis('off')

    plt.suptitle(f'bbox 보존 검증 | 원본 {len(bboxes)}개 → 증강 후 {len(aug_bboxes)}개', fontsize=12)
    plt.tight_layout()
    plt.show()

verify_bbox_after_aug(image_id=sample_ids[1])

---
## 요약

| 항목 | 권장값 | 근거 |
|------|--------|------|
| 입력 해상도 | 1024 × 1024 | mAP@0.75+ 기준, 작은 객체 디테일 보존 |
| 정규화 | mean=[0.485,0.456,0.406] / std=[0.229,0.224,0.225] | ImageNet 사전학습 기준 |
| Augmentation | `train_transform` 파이프라인 참고 | 환경 편향 + 반쪽 알약 대응 |
| bbox 포맷 | COCO (x, y, w, h) | 제출 포맷과 동일 |
| min_visibility | 0.3 | 잘린 bbox 필터링 기준 |

```python
#  DataLoader 연결 예시
from torch.utils.data import DataLoader

train_dataset = PillDataset(coco, TRAIN_IMG_DIR, transform=train_transform)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
```
